# Introdução

O controlador Proporcional-Integral-Derivativo (PID) representa a arquitetura de controle em malha fechada mais utilizada no mundo. Desde a sua formalização matemática inicial para o direcionamento automático de embarcações, o PID estabeleceu-se como o padrão de fato da automação industrial devido ao seu equilíbrio ideal entre **eficiência de desempenho, robustez e simplicidade conceitual**. Ele funciona da seguinte maneira:

- Ação Proporcional ($K_p$): Atua no presente, fornecendo uma resposta direta e proporcional ao erro instantâneo $e(t) = r(t) - y(t)$.
  
- Ação Integral ($K_i$): Atua no histórico acumulado do erro, eliminando o erro em regime permanente (*offset*) inerente a perturbações de carga.
  
- Ação Derivativa ($K_d$): Prevê a tendência futura do erro avaliando sua taxa de variação $\frac{de(t)}{dt}$, introduzindo amortecimento e aumentando a estabilidade transitória.

Embora a maioria dos processos industriais reais exiba comportamentos não lineares, atrasos de transporte e acoplamentos dinâmicos, um controlador PID bem sintonizado consegue estabilizar e operar com excelente desempenho a grande maioria das plantas em torno de seus pontos de operação nominais.

Apesar disso, o gargalo no uso do PID reside na escolha ideal dos ganhos ($K_p, K_i, K_d$). É exatamente nessa lacuna que a **Programação Não Linear (PNL)** e os conceitos de **Tradeoff e Regularização** se tornam indispensáveis. Ao formular a sintonia como um problema de otimização contínua restrita

$$\min_{K_p, K_i, K_d} J(K_p, K_i, K_d) = \int_0^T t \cdot |e(t)| \, dt + \lambda \int_0^T u(t)^2 \, dt$$

# Modelo Matemático

### 2.1 Estrutura da Malha Fechada
Estabelecendo a derivação matemática rigorosa do problema de otimização, explicitando as hipóteses de modelagem, a formalização das variáveis de decisão, a função objetivo ponderada e o conjunto de restrições operacionais.

Considera-se um sistema dinâmico linear e invariante no tempo (LTI) representado pela sua função de transferência em tempo contínuo $G(s) = \frac{Y(s)}{U(s)}$, onde $Y(s)$ é a saída controlada e $U(s)$ é o sinal de atuação.

Para a aplicação prática, adota-se o modelo clássico de posicionamento/velocidade de um Motor DC com carga, cuja dinâmica nominal de segunda ordem é dada por:

$$G(s) = \frac{K}{s(\tau s + 1)} = \frac{Y(s)}{U(s)}$$

em que $K$ é o ganho estático e $\tau$ é a constante de tempo do sistema. No domínio do tempo, a relação diferencial entre a atuação $u(t)$ e a resposta $y(t)$ é descrita pela equação diferencial:

$$\tau \ddot{y}(t) + \dot{y}(t) = K u(t)$$

O controlador PID opera realimentando o erro de rastreamento $e(t) = r(t) - y(t)$, em que $r(t)$ é o sinal de referência (degrau unitário). A lei de controle em tempo contínuo é expressa por:

$$u(t) = K_p e(t) + K_i \int_{0}^{t} e(\tau) \, d\tau + K_d \frac{de(t)}{dt}$$

---

### 2.2 Variáveis de Decisão e Domínio de Busca

O objetivo da otimização é determinar os ganhos ótimos do controlador PID. Assim, define-se o vetor de variáveis de decisão $\boldsymbol{x} \in \mathbb{R}^3$ como:

$$\boldsymbol{x} = \begin{bmatrix} K_p \\ K_i \\ K_d \end{bmatrix}$$

Para garantir a estabilidade do sistema em malha fechada e respeitar os limites operacionais físicos dos circuitos ou do algoritmo de controle, impõem-se limites de caixa  inferiores e superiores sobre as variáveis de decisão:

$$  K_{p,\mathrm{min}} \le K_p \le K_{p,\mathrm{max}}$$
$$ K_{i,\mathrm{min}} \le K_i \le K_{i,\mathrm{max}}, $$
$$ K_{d,\mathrm{min}} \le K_d \le K_{d,\mathrm{max}} $$

---

### 2.3 Formulação da Função Custo Multiobjetivo (Tradeoff e Regularização)
A sintonia de controladores envolve inerentemente dois objetivos conflitantes:

1. **Desempenho Temporal ($J_1$)**: Minimizar o erro acumulado entre a resposta $y(t)$ e a referência $r(t)$.
2. **Esforço de Atuação ($J_2$)**: Minimizar a energia do sinal de controle $u(t)$ para evitar a saturação e o desgaste do atuador.

Para quantificar o **desempenho temporal**, adota-se o critério **ITAE** (*Integral of Time-weighted Absolute Error*), que penaliza erros que persistem ao longo do tempo:

$$J_1(\boldsymbol{x}) =\int_{0}^{T_{sim}} t \cdot |e(t, \boldsymbol{x})| \, dt$$

Para quantificar a **energia do sinal de controle** e introduzir **suavidade na resposta da atuação**, adota-se a **Regularização $L_2$ (Tikhonov / Ridge)**:

$$J_2(\boldsymbol{x}) = \int_{0}^{T_{\mathrm{sim}}} [u(t, \boldsymbol{x})]^2 \, \mathrm{d}t = \Vert{}u(\cdot, \boldsymbol{x})\Vert{}_{\mathcal{L}_2}^2$$

Formula-se o problema multiobjetivo combinando ambos os critérios em uma função custo escalar parametrizada pelo peso de regularização $\lambda > 0$:

$$\min_{\boldsymbol{x} \in \Omega_{\boldsymbol{x}}} J(\boldsymbol{x}; \lambda) = J_1(\boldsymbol{x}) + \lambda J_2(\boldsymbol{x}) = \int_{0}^{T_{sim}} \left( t \cdot \vert{}e(t, \boldsymbol{x})\vert{} + \lambda u(t, \boldsymbol{x})^2 \right) dt$$

A variação contínua do parâmetro de regularização $\lambda \in (0, \infty)$ permite mapear a Fronteira Pareto-ótima (Curva de Pareto) no plano $J_2(\boldsymbol{x}_\lambda^*)$ versus $J_1(\boldsymbol{x}_\lambda^*)$, evidenciando o *trade-off* entre precisão e consumo de energia.

---

### 2.4 Discretização Numérica e Integração Temporal
Como as trajetórias temporais $e(t, \boldsymbol{x})$ e $u(t, \boldsymbol{x})$ são obtidas por simulação numérica em um horizonte $T_{\mathrm{sim}}$ com passo de amostragem $\Delta t$, a integral contínua é aproximada pela regra dos trapézios com $N = \frac{T_{\mathrm{sim}}}{\Delta t}$ pontos de amostragem ($t_k = k \cdot \Delta t$):

$$J_1(\boldsymbol{x}) \approx \sum_{k=0}^{N-1} \frac{t_k \vert{}e(t_k, \boldsymbol{x})\vert{} + t_{k+1} \vert{}e(t_{k+1}, \boldsymbol{x})\vert{}}{2} \Delta t$$

$$J_2(\boldsymbol{x}) \approx \sum_{k=0}^{N-1} \frac{[u(t_k, \boldsymbol{x})]^2 + [u(t_{k+1}, \boldsymbol{x})]^2}{2} \Delta t$$

A avaliação de $J(\boldsymbol{x}; \lambda)$ exige a resolução da equação diferencial de malha fechada a cada iteração do otimizador.


--- 

### 2.5

# Solução Computacional

# Resultados e Análise Gráfica

# Discussão